# Parte 7 — Polars: uma alternativa moderna ao pandas

**Objetivo desta parte:** até aqui, todo o tratamento e a agregação de dados no
projeto foram feitos com `pandas` (Partes [2](02_tratamento_pandas.ipynb) e
[3](03_nova_visao_agregacoes.ipynb)), que é exatamente o que
[`src/clima_pipeline`](../src/clima_pipeline) usa em produção. Este notebook é um
**desvio comparativo**, não uma continuação do pipeline: vamos conhecer o
[Polars](https://pola.rs/), uma biblioteca de DataFrames mais nova, escrita em
Rust, e entender — usando os mesmos dados já tratados em `data/processed/` — onde
ela se parece com o pandas, onde a sintaxe diverge de propósito, e onde ela ganha
(ou não) em desempenho.

Nada aqui substitui o pandas no pacote de produção deste projeto — a ideia é
ampliar o repertório e saber reconhecer quando o Polars seria a escolha certa em
outro contexto.

## Por que o Polars existe

O pandas foi criado em 2008 sobre NumPy, roda em uma única thread por padrão e
sempre executa cada operação assim que ela é chamada (*eager evaluation*). Isso é
simples de raciocinar, mas deixa desempenho na mesa em datasets grandes: não há
paralelismo automático, e cada método intermediário de uma cadeia gera um
DataFrame novo na memória, mesmo que colunas inteiras nunca cheguem a ser usadas.

O [Polars](https://pola.rs/) nasceu em 2020 para atacar exatamente isso:

- **Escrito em Rust**, sem o overhead do interpretador Python no caminho crítico
  de execução — a API Python é uma casca fina sobre a implementação nativa.
- **Paralelismo automático entre colunas e linhas**, usando todos os núcleos
  disponíveis sem que o usuário precise pedir (nada de
  `multiprocessing`/`joblib` manual).
- **Baseado em [Apache Arrow](https://arrow.apache.org/)**, um formato de memória
  colunar padronizado — o que facilita interoperar com outras ferramentas do
  ecossistema (DuckDB, Spark, ferramentas de ML) sem serializar/desserializar.
- **API de expressões** (`pl.col(...)`) que descreve *o que* calcular sem
  amarrar a um jeito específico de executar — a mesma expressão roda tanto no
  modo eager quanto no modo lazy (mais sobre isso na seção 6).
- **Modo lazy opcional**, em que uma cadeia de operações só é de fato executada
  no final, depois de um otimizador reorganizar o plano (seção 6) — algo que o
  pandas simplesmente não tem.

Instalação (já feita neste ambiente): `pip install polars`.

In [2]:
from pathlib import Path
import time

import pandas as pd
import polars as pl

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

PROCESSED_DIR = Path("../data/processed")

pandas: 2.3.3
polars: 1.44.1


## 1. Lendo os mesmos dados com as duas bibliotecas

Vamos carregar `clima_tratado.csv` — o resultado da Parte 2, já limpo — com as
duas bibliotecas lado a lado. A sintaxe de leitura já é a primeira pista de como
o Polars pensa diferente: em vez de `parse_dates=[...]`, ele expõe
`try_parse_dates=True`, que inspecciona as colunas de texto e converte as que
*parecem* data/hora — não precisa listar cada coluna por nome.

In [3]:
df_pd = pd.read_csv(PROCESSED_DIR / "clima_tratado.csv", parse_dates=["datetime"])
df_pl = pl.read_csv(PROCESSED_DIR / "clima_tratado.csv", try_parse_dates=True)

print(df_pd.shape, df_pl.shape)
df_pd.head(3)

(3720, 6) (3720, 6)


,cidade,datetime,temp_c,umidade_pct,precipitacao_mm,vento_kmh
0,manaus,2025-01-01 00:00:00,26.7,90.0,0.0,6.4
1,manaus,2025-01-01 01:00:00,26.6,91.0,0.0,5.6
2,manaus,2025-01-01 02:00:00,26.6,92.0,0.0,6.5


In [19]:
df_pl.head(3)

cidade,datetime,temp_c,umidade_pct,precipitacao_mm,vento_kmh
str,datetime[μs],f64,f64,f64,f64
"""manaus""",2025-01-01 00:00:00,26.7,90.0,0.0,6.4
"""manaus""",2025-01-01 01:00:00,26.6,91.0,0.0,5.6
"""manaus""",2025-01-01 02:00:00,26.6,92.0,0.0,6.5


### A diferença mais estrutural: não existe índice

Repare que a saída do Polars não tem a coluna de índice (`0, 1, 2, ...`) à
esquerda que o pandas sempre mostra. Isso não é só estética — o pandas usa o
índice para alinhar dados automaticamente em operações (soma entre duas Series,
por exemplo, casa por índice antes de somar), o que é poderoso mas também fonte
de bugs sutis quando o índice de duas fontes não bate. O Polars **não tem
índice**: toda linha é identificada apenas pela posição no momento, e operações
entre colunas sempre casam por posição, nunca por um rótulo escondido.

Consequência prática: em pandas, filtrar ou ordenar preserva os rótulos antigos
do índice (por isso é comum ver `.reset_index(drop=True)` depois de um filtro);
em Polars isso não existe — não há o que resetar.

Outra diferença visível: `df.dtypes` no pandas devolve uma `Series` (rótulo por
coluna); no Polars devolve uma lista simples, na mesma ordem das colunas de
`df.columns`.

In [5]:
print("pandas dtypes:")
print(df_pd.dtypes)
print()
print("polars dtypes:", df_pl.dtypes)

pandas dtypes:
cidade                     object
datetime           datetime64[ns]
temp_c                    float64
umidade_pct               float64
precipitacao_mm           float64
vento_kmh                 float64
dtype: object

polars dtypes: [String, Datetime(time_unit='us', time_zone=None), Float64, Float64, Float64, Float64]


## 2. Seleção e filtragem: a API de expressões (`pl.col`)

Em pandas, selecionar colunas e filtrar linhas usa colchetes e máscaras
booleanas construídas a partir do próprio DataFrame (`df[df["temp_c"] > 32]`). O
Polars separa isso em dois métodos explícitos — `.select()` para colunas e
`.filter()` para linhas — e descreve a condição com uma **expressão**
(`pl.col("temp_c") > 32`), um objeto que representa o cálculo sem executá-lo
ainda.

In [6]:
# pandas
horas_quentes_pd = df_pd[df_pd["temp_c"] > 32][["cidade", "datetime", "temp_c"]]

# polars
horas_quentes_pl = df_pl.filter(pl.col("temp_c") > 32).select("cidade", "datetime", "temp_c")

print(len(horas_quentes_pd), len(horas_quentes_pl))
horas_quentes_pl.head(3)

114 114


cidade,datetime,temp_c
str,datetime[μs],f64
"""porto_alegre""",2025-01-02 14:00:00,32.4
"""porto_alegre""",2025-01-14 14:00:00,32.4
"""porto_alegre""",2025-01-14 15:00:00,32.7


`pl.col("temp_c") > 32` não olha para nenhum DataFrame em particular — é só uma
receita ("pegue a coluna chamada `temp_c` e compare com 32"). Isso é o que
permite a mesma expressão ser reaproveitada em `.filter()`, `.select()`,
`.with_columns()` e `.group_by(...).agg()`, e é também a peça que o motor lazy
usa para otimizar o plano inteiro antes de rodar (seção 6).

## 3. Criando colunas: `with_columns` vs. atribuição/`assign`

Em pandas, criar uma coluna nova é `df["nova"] = ...` (efeito colateral no
DataFrame) ou `df.assign(nova=...)` (devolve uma cópia, sem alterar o original).
Em Polars só existe o segundo estilo, via `.with_columns(...)` — e ele aceita
**várias expressões de uma vez**, cada uma calculada em paralelo sobre o mesmo
DataFrame de entrada.

In [7]:
# pandas — duas colunas novas, duas atribuições
df_pd_novo = df_pd.assign(
    temp_f=lambda d: d["temp_c"] * 9 / 5 + 32,
    vento_ms=lambda d: d["vento_kmh"] / 3.6,
)

# polars — as duas expressões rodam na mesma chamada
df_pl_novo = df_pl.with_columns(
    (pl.col("temp_c") * 9 / 5 + 32).alias("temp_f"),
    (pl.col("vento_kmh") / 3.6).alias("vento_ms"),
)

df_pl_novo.select("cidade", "temp_c", "temp_f", "vento_kmh", "vento_ms").head(3)

cidade,temp_c,temp_f,vento_kmh,vento_ms
str,f64,f64,f64,f64
"""manaus""",26.7,80.06,6.4,1.777778
"""manaus""",26.6,79.88,5.6,1.555556
"""manaus""",26.6,79.88,6.5,1.805556


Note o `.alias(...)` no lugar do `nome=` do `assign` — como a expressão em si
não carrega nome de coluna (ela só descreve um cálculo), é preciso nomeá-la
explicitamente para usá-la num `with_columns`/`select`.

## 4. Agregações: `group_by` + `agg` vs. `groupby` + `agg`

A sintaxe de agrupar e agregar é onde pandas e Polars mais se parecem — e onde a
diferença de filosofia aparece com mais clareza. Reaproveitando o mesmo cálculo
da [Parte 3](03_nova_visao_agregacoes.ipynb) (estatísticas de temperatura por
cidade):

In [8]:
# pandas
resumo_pd = (
    df_pd.groupby("cidade")["temp_c"]
    .agg(temp_media="mean", temp_min="min", temp_max="max")
    .sort_values("temp_media", ascending=False)
)

resumo_pd

,temp_media,temp_min,temp_max
cidade,,,
manaus,27.070766,23.4,31.7
rio_de_janeiro,26.992339,21.5,38.0
recife,26.910820,22.9,31.6
porto_alegre,25.181048,18.1,36.3
sao_paulo,22.740054,15.4,33.5


In [9]:
# polars
resumo_pl = (
    df_pl.group_by("cidade")
    .agg(
        pl.col("temp_c").mean().alias("temp_media"),
        pl.col("temp_c").min().alias("temp_min"),
        pl.col("temp_c").max().alias("temp_max"),
        pl.len().alias("n_registros"),
    )
    .sort("temp_media", descending=True)
)

resumo_pl

cidade,temp_media,temp_min,temp_max,n_registros
str,f64,f64,f64,u32
"""manaus""",27.070766,23.4,31.7,744
"""rio_de_janeiro""",26.992339,21.5,38.0,744
"""recife""",26.91082,22.9,31.6,744
"""porto_alegre""",25.181048,18.1,36.3,744
"""sao_paulo""",22.740054,15.4,33.5,744


Duas diferenças que valem a pena guardar:

- `agg(...)` em pandas recebe **strings** com o nome do método (`"mean"`) ou uma
  função; em Polars recebe **expressões** (`pl.col("temp_c").mean()`), que podem
  ser combinadas e nomeadas livremente na mesma chamada — dá para agregar várias
  colunas com transformações diferentes num único `.agg(...)`, sem precisar de
  um dicionário de listas como em `df.groupby(...).agg({"a": ["mean"], "b": ["sum"]})`.
- Por padrão, `df.groupby(...)` em pandas ordena as chaves do grupo
  (`sort=True`); `df.group_by(...)` em Polars **não garante ordem alguma**, a
  menos que se passe `maintain_order=True` — o motor pode processar os grupos na
  ordem que for mais rápida. Se a ordem importa (por exemplo, para bater com a
  ordem de outra tabela), a forma idiomática é ordenar explicitamente depois,
  como fizemos acima com `.sort("temp_media", descending=True)`.

## 5. Janelas por grupo: `.rolling()` + `groupby` vs. `.over()`

A Parte 3 calcula uma média móvel de 3 dias por cidade
(`media_movel_3d`, em [`clima_diario.csv`](../data/processed/clima_diario.csv))
combinando `groupby` com `.rolling()`. Vamos reproduzir o mesmo cálculo com as
duas bibliotecas, a partir do resumo diário já pronto.

In [10]:
diario_pd = pd.read_csv(PROCESSED_DIR / "clima_diario.csv", parse_dates=["data"])
diario_pl = pl.read_csv(PROCESSED_DIR / "clima_diario.csv", try_parse_dates=True)

base_pd = diario_pd[["cidade", "data", "temp_media"]].sort_values(["cidade", "data"])
base_pl = diario_pl.select("cidade", "data", "temp_media").sort("cidade", "data")

In [11]:
# pandas — groupby + transform com uma rolling window
base_pd = base_pd.assign(
    media_movel_3d=base_pd.groupby("cidade")["temp_media"]
    .transform(lambda serie: serie.rolling(window=3).mean())
)

base_pd[base_pd["cidade"] == "manaus"].head(5)

,cidade,data,temp_media,media_movel_3d
0,manaus,2025-01-01,27.391667,NaN
1,manaus,2025-01-02,25.908333,NaN
2,manaus,2025-01-03,27.587500,26.962500
3,manaus,2025-01-04,26.233333,26.576389
4,manaus,2025-01-05,25.795833,26.538889


In [12]:
# polars — a expressão de rolling roda "sobre" (.over) cada grupo de cidade
base_pl = base_pl.with_columns(
    pl.col("temp_media").rolling_mean(window_size=3).over("cidade").alias("media_movel_3d")
)

base_pl.filter(pl.col("cidade") == "manaus").head(5)

cidade,data,temp_media,media_movel_3d
str,date,f64,f64
"""manaus""",2025-01-01,27.391667,null
"""manaus""",2025-01-02,25.908333,null
"""manaus""",2025-01-03,27.5875,26.9625
"""manaus""",2025-01-04,26.233333,26.576389
"""manaus""",2025-01-05,25.795833,26.538889


`.over("cidade")` é o equivalente Polars de uma *window function* de SQL
(`AVG(...) OVER (PARTITION BY cidade ORDER BY data ROWS 2 PRECEDING)`): a
expressão à esquerda é calculada **dentro de cada grupo**, mas o resultado tem o
mesmo número de linhas da tabela original — diferente de `group_by().agg()`, que
colapsa cada grupo numa única linha. É uma alternativa mais direta ao par
`groupby(...).transform(...)` do pandas, e generaliza para qualquer expressão
(não só `.rolling_mean()`).

## 6. `join` vs. `merge`

Igual à Parte 3, vamos enriquecer os dados diários com uma tabela pequena de
metadados por cidade (UF e região) e cruzar as duas.

In [13]:
cidades_meta = [
    {"cidade": "sao_paulo", "nome_exibicao": "São Paulo", "uf": "SP", "regiao": "Sudeste"},
    {"cidade": "rio_de_janeiro", "nome_exibicao": "Rio de Janeiro", "uf": "RJ", "regiao": "Sudeste"},
    {"cidade": "manaus", "nome_exibicao": "Manaus", "uf": "AM", "regiao": "Norte"},
    {"cidade": "porto_alegre", "nome_exibicao": "Porto Alegre", "uf": "RS", "regiao": "Sul"},
    {"cidade": "recife", "nome_exibicao": "Recife", "uf": "PE", "regiao": "Nordeste"},
]

cidades_meta_pd = pd.DataFrame(cidades_meta)
cidades_meta_pl = pl.DataFrame(cidades_meta)

# pandas
enriquecido_pd = base_pd.merge(cidades_meta_pd, on="cidade", how="left")

# polars
enriquecido_pl = base_pl.join(cidades_meta_pl, on="cidade", how="left")

enriquecido_pl.head(3)

cidade,data,temp_media,media_movel_3d,nome_exibicao,uf,regiao
str,date,f64,f64,str,str,str
"""manaus""",2025-01-01,27.391667,null,"""Manaus""","""AM""","""Norte"""
"""manaus""",2025-01-02,25.908333,null,"""Manaus""","""AM""","""Norte"""
"""manaus""",2025-01-03,27.5875,26.9625,"""Manaus""","""AM""","""Norte"""


O nome do método muda (`merge` → `join`) e o argumento de tipo de junção também
é levemente diferente (`how="left"` em ambos, mas Polars usa `"inner"`/`"left"`/
`"full"`/`"semi"`/`"anti"` — os dois últimos, *semi* e *anti* join, não têm
equivalente direto de uma linha em pandas: filtram a tabela da esquerda mantendo
só as linhas que **têm** (semi) ou **não têm** (anti) correspondência na direita,
sem trazer nenhuma coluna da direita).

## 7. Eager vs. Lazy: a diferença que o pandas não tem

Tudo até aqui foi **eager** nas duas bibliotecas: cada linha de código executa
imediatamente. O Polars tem um segundo modo — **lazy** — em que, em vez de
`pl.read_csv`, usamos `pl.scan_csv`: isso devolve um `LazyFrame`, que não lê o
arquivo ainda, só registra a intenção. Cada `.filter()`/`.select()`/`.group_by()`
encadeado apenas acrescenta um passo ao **plano de consulta**; nada roda de fato
até chamarmos `.collect()`.

A vantagem é que, antes de executar, o otimizador do Polars reorganiza o plano —
por exemplo, empurrando o filtro para dentro da leitura do CSV
(*predicate pushdown*, para não desperdiçar tempo decodificando linhas que serão
descartadas) e lendo do disco só as colunas realmente usadas
(*projection pushdown*). Em pandas isso não existe: `pd.read_csv` sempre lê o
arquivo inteiro, e cada `.query()`/`.groupby()` seguinte processa o resultado
completo do passo anterior.

In [14]:
consulta = (
    pl.scan_csv(PROCESSED_DIR / "clima_tratado.csv", try_parse_dates=True)
    .filter(pl.col("temp_c") > 25)
    .group_by("cidade")
    .agg(pl.col("temp_c").mean().alias("temp_media"))
    .sort("temp_media", descending=True)
)

print(consulta.explain())

SORT BY [descending: [true]] [col("temp_media")]
  AGGREGATE[maintain_order: false]
    [col("temp_c").mean().alias("temp_media")] BY [col("cidade")]
    FROM
    Csv SCAN [../data/processed/clima_tratado.csv]
    PROJECT 2/6 COLUMNS
    SELECTION: col("temp_c") > 25.0
    ESTIMATED ROWS: 4032


Lendo o plano de baixo para cima: `Csv SCAN` já aparece com `PROJECT 2/6 COLUMNS`
(só `cidade` e `temp_c` chegam a ser lidas, mesmo o CSV tendo 6 colunas) e
`SELECTION: col("temp_c") > 25.0` (o filtro é aplicado durante a própria leitura,
não depois). Só quando chamamos `.collect()` é que a consulta de fato roda:

In [15]:
consulta.collect()

cidade,temp_media
str,f64
"""rio_de_janeiro""",28.488314
"""porto_alegre""",28.352273
"""sao_paulo""",27.738947
"""manaus""",27.365841
"""recife""",27.36564


Regra prática: comece com `pl.scan_csv` sempre que a fonte for um arquivo (ou
vários) e a consulta terminar num resultado agregado/filtrado bem menor que a
entrada — é o caso mais comum de pipeline de dados. Use `pl.read_csv` (eager)
quando você precisa inspecionar o DataFrame inteiro interativamente, como
estivemos fazendo nas seções anteriores deste notebook.

## 8. Desempenho: um benchmark simples

`clima_tratado.csv` tem só ~3.700 linhas — pequeno demais para qualquer biblioteca
suar. Para tornar a comparação honesta, replicamos os dados 270 vezes (~1 milhão
de linhas, mantendo os mesmos valores — o objetivo aqui é só medir a agregação
em si, não analisar os números) e cronometramos a mesma agregação por cidade nas
duas bibliotecas.

In [16]:
pd_grande = pd.concat([df_pd] * 270, ignore_index=True)
pl_grande = pl.concat([df_pl] * 270)

print(f"{len(pd_grande):,} linhas".replace(",", "."))

1.004.400 linhas


In [17]:
inicio = time.perf_counter()
_ = pd_grande.groupby("cidade")["temp_c"].agg(["mean", "min", "max"])
tempo_pandas = time.perf_counter() - inicio

inicio = time.perf_counter()
_ = pl_grande.group_by("cidade").agg(
    pl.col("temp_c").mean().alias("mean"),
    pl.col("temp_c").min().alias("min"),
    pl.col("temp_c").max().alias("max"),
)
tempo_polars = time.perf_counter() - inicio

print(f"pandas: {tempo_pandas * 1000:.1f} ms")
print(f"polars: {tempo_polars * 1000:.1f} ms")
print(f"polars foi {tempo_pandas / tempo_polars:.1f}x mais rápido nesta máquina")

pandas: 51.3 ms
polars: 12.1 ms
polars foi 4.2x mais rápido nesta máquina


O número exato varia por máquina e por quantas vezes você rodar a célula (a
primeira execução costuma pagar um custo de *warm-up*), mas a ordem de grandeza
— Polars alguns múltiplos mais rápido no mesmo `group_by`/`agg` — é consistente
com o que a documentação oficial reporta em benchmarks maiores
([H2O.ai db-benchmark](https://h2oai.github.io/db-benchmark/), mantido pela
comunidade). A causa é a soma de tudo visto até aqui: paralelismo automático
entre os grupos, sem overhead de laço em Python, sobre um formato de memória
colunar pensado para isso.

Vale o contraponto: para os ~3.700 registros reais deste projeto, essa diferença
é irrelevante em termos absolutos (a operação inteira leva menos de 1 ms nas
duas bibliotecas) — o ganho do Polars só importa a partir de volumes bem
maiores que os deste dataset.

## 9. Interoperabilidade: convertendo entre as duas

Como as duas bibliotecas usam Arrow (ou conseguem falar Arrow) por baixo, ir e
voltar entre elas é direto — útil, por exemplo, para aproveitar uma função que só
existe no ecossistema pandas (`matplotlib`/`seaborn` via `.plot()`, `scikit-learn`)
depois de tratar os dados em Polars.

In [18]:
de_volta_pandas = resumo_pl.to_pandas()
de_volta_polars = pl.from_pandas(resumo_pd.reset_index())

print(type(de_volta_pandas), type(de_volta_polars))
de_volta_pandas

<class 'pandas.core.frame.DataFrame'> <class 'polars.dataframe.frame.DataFrame'>


,cidade,temp_media,temp_min,temp_max,n_registros
0,manaus,27.070766,23.4,31.7,744
1,rio_de_janeiro,26.992339,21.5,38.0,744
2,recife,26.910820,22.9,31.6,744
3,porto_alegre,25.181048,18.1,36.3,744
4,sao_paulo,22.740054,15.4,33.5,744


## 10. Resumo: sintaxe lado a lado

| Operação | pandas | Polars |
|---|---|---|
| Ler CSV | `pd.read_csv(caminho, parse_dates=[...])` | `pl.read_csv(caminho, try_parse_dates=True)` |
| Ler CSV (preguiçoso) | *(não existe)* | `pl.scan_csv(caminho)` → `LazyFrame` |
| Selecionar colunas | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filtrar linhas | `df[df["a"] > 1]` | `df.filter(pl.col("a") > 1)` |
| Nova coluna | `df["c"] = ...` / `df.assign(c=...)` | `df.with_columns(expr.alias("c"))` |
| Agrupar + agregar | `df.groupby("a").agg(...)` | `df.group_by("a").agg(...)` |
| Ordenar | `df.sort_values("a")` | `df.sort("a")` |
| Cruzar tabelas | `df.merge(outro, on="a", how="left")` | `df.join(outro, on="a", how="left")` |
| Janela por grupo | `df.groupby("a")["b"].transform(...)` | `pl.col("b").expr().over("a")` |
| Índice de linha | `df.index` (rótulos, usado para alinhar) | não existe |
| Execução | sempre eager | eager (`read_*`) ou lazy (`scan_*` + `.collect()`) |
| Paralelismo | manual (`multiprocessing`, `dask`) | automático |

## Quando escolher cada um

- **pandas** continua sendo a escolha certa para este projeto: o dataset final
  tem poucos milhares de linhas, e o pandas já integra diretamente com
  `SQLAlchemy` (`to_sql`/`read_sql`, Parte [4](05_sqlite_persistencia.ipynb)),
  `matplotlib`/`seaborn` (Parte [3.2](04_exploracao_visual.ipynb)) e o
  ecossistema de ML (`scikit-learn`), sem nenhuma camada de conversão. É também
  o que a maior parte dos tutoriais, Stack Overflow e colegas de equipe já
  conhecem.
- **Polars** compensa quando o volume de dados cresce (dezenas de milhões de
  linhas ou mais), quando o pipeline já é uma cadeia de transformações que se
  beneficia do otimizador do modo lazy, ou quando o processo roda em lote,
  agendado, sem ninguém olhando — cenário em que desempenho bruto importa mais
  do que familiaridade da sintaxe